# 2) Adapter Training (Colab)

Headless continual adapter training with Drive-based artifact output.

In [ ]:
from google.colab import drive, userdata

import os
import subprocess
import sys

drive.mount('/content/drive')

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")

def _resolve_hf_token() -> str | None:
    for env_name in HF_TOKEN_NAMES:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault("HF_TOKEN", token)
            return token
    for secret_name in HF_TOKEN_NAMES:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    return None

hf_token = _resolve_hf_token()
if not hf_token:
    print("[HF] No token found. Set a Colab secret or env var named HF_TOKEN before training.")
else:
    try:
        from huggingface_hub import HfApi, login
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
        from huggingface_hub import HfApi, login

    try:
        login(token=hf_token, add_to_git_credential=False)
        profile = dict(HfApi(token=hf_token).whoami() or {})
        username = str(
            profile.get("name")
            or profile.get("fullname")
            or profile.get("email")
            or profile.get("user")
            or "authenticated user"
        )
        print(f"[HF] Authenticated as {username}")
    except Exception as exc:
        print(f"[HF] Authentication check failed: {exc}")
        print("[HF] Training may fail when gated models need authentication.")

In [ ]:
import io
import json
import os
import random
import shutil
import sys
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: resolve repo root, install deps ---
try:
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )
except ModuleNotFoundError:
    # Colab: auto-clone if repo is not on path
    _clone_target = Path("/content/bitirmeprojesi")
    if not (_clone_target / "scripts").exists():
        import subprocess

        _url = os.environ.get("AADS_REPO_URL", "https://github.com/EfeErim/bitirmeprojesi.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_clone_target)], check=True)
    os.chdir(_clone_target)
    sys.path.insert(0, str(_clone_target))
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / "colab_notebooks" / "requirements_colab.txt", IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, "capture_cell_output"):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(value or "").strip())
        while "__" in slug:
            slug = slug.replace("__", "_")
        slug = slug.strip("_")
        return slug or "cell"

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, "flush", None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, "isatty", lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

        @property
        def encoding(self):
            for stream in self._streams:
                encoding = getattr(stream, "encoding", None)
                if encoding:
                    return str(encoding)
            return "utf-8"

        def __getattr__(self, name):
            if not self._streams:
                raise AttributeError(name)
            return getattr(self._streams[0], name)

    @contextmanager
    def _capture_cell_output(self, cell_name: str, *, phase: str = "notebook_cell"):
        label = str(cell_name or "cell")
        capture_ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        started_at = datetime.utcnow().isoformat() + "Z"
        start_clock = time.time()
        relative_path = Path("cell_outputs") / f"{capture_ts}_{_slugify_capture_name(label)}.log"
        relative_path_text = str(relative_path).replace("\\", "/")
        stdout_buffer = io.StringIO()
        stderr_buffer = io.StringIO()
        original_stdout = sys.stdout
        original_stderr = sys.stderr
        failure_message = ""

        emit_event = getattr(self, "emit_event", None)
        if callable(emit_event):
            emit_event(
                "cell_capture_started",
                {"cell_name": label, "relative_path": relative_path_text},
                phase=phase,
                force_sync=False,
            )

        sys.stdout = _CompatTeeTextStream(original_stdout, stdout_buffer)
        sys.stderr = _CompatTeeTextStream(original_stderr, stderr_buffer)
        try:
            yield self.artifacts_dir / relative_path
        except Exception as exc:
            failure_message = f"{exc.__class__.__name__}: {exc}"
            if callable(emit_event):
                emit_event(
                    "cell_capture_failed",
                    {
                        "cell_name": label,
                        "relative_path": relative_path_text,
                        "error": failure_message,
                    },
                    phase=phase,
                    level="error",
                    force_sync=False,
                )
            raise
        finally:
            sys.stdout = original_stdout
            sys.stderr = original_stderr
            duration_sec = round(time.time() - start_clock, 3)
            finished_at = datetime.utcnow().isoformat() + "Z"
            body = "\n".join(
                [
                    f"[CELL_CAPTURE] cell={label}",
                    f"run_id={getattr(self, 'run_id', '')}",
                    f"started_at={started_at}",
                    f"finished_at={finished_at}",
                    f"duration_sec={duration_sec:.3f}",
                    "",
                    stdout_buffer.getvalue(),
                ]
            )
            stderr_text = stderr_buffer.getvalue()
            if stderr_text:
                if not body.endswith("\n"):
                    body += "\n"
                body += "\n[STDERR]\n" + stderr_text
            if failure_message:
                if not body.endswith("\n"):
                    body += "\n"
                body += f"\n[EXCEPTION] {failure_message}\n"

            write_text_artifact = getattr(self, "write_text_artifact", None)
            if callable(write_text_artifact):
                artifact_path = write_text_artifact(relative_path_text, body)
            else:
                artifact_path = self.artifacts_dir / relative_path
                artifact_path.parent.mkdir(parents=True, exist_ok=True)
                artifact_path.write_text(body, encoding="utf-8")

            if callable(emit_event):
                emit_event(
                    "cell_capture_finished",
                    {
                        "cell_name": label,
                        "relative_path": relative_path_text,
                        "artifact_path": str(artifact_path),
                        "duration_sec": duration_sec,
                    },
                    phase=phase,
                    force_sync=False,
                )

            periodic_sync = getattr(self, "_periodic_sync", None)
            if callable(periodic_sync):
                periodic_sync(force=True)

    ColabLiveTelemetry.capture_cell_output = _capture_cell_output
    return True


_CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()

# --- Device ---
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime required. In Colab: Runtime -> Change runtime type -> GPU."
    )
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# --- Telemetry + checkpoint manager ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_ROOT = Path(os.environ.get("AADS_DRIVE_LOG_ROOT", "/content/drive/MyDrive/aads_ulora"))
CHECKPOINT_MANAGER = TrainingCheckpointManager(DRIVE_ROOT / "telemetry" / RUN_ID, retention=3)
TELEMETRY = ColabLiveTelemetry(
    notebook_name="2_interactive_adapter_training",
    run_id=RUN_ID,
    drive_root=DRIVE_ROOT,
)

def rt(message: str, *, phase: str = "notebook", level: str = "info") -> None:
    print(message)
    TELEMETRY.emit_log(message, phase=phase, level=level)

setup_message = f"[SETUP] root={ROOT}  device={DEVICE}  run_id={RUN_ID}"
print(setup_message)
if _CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED:
    print("[SETUP] Installed notebook-local capture_cell_output compatibility shim.")
print(f"[SETUP] cell outputs -> {TELEMETRY.artifacts_dir / 'cell_outputs'}")
TELEMETRY.emit_log(setup_message, phase="setup")
TELEMETRY.update_latest(
    {
        "phase": "setup",
        "device": DEVICE,
        "root": str(ROOT),
        "run_id": RUN_ID,
        "capture_cell_output_compat": bool(_CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED),
    }
)

In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Training parameters ---
    # Edit these directly, then run the remaining cells top-to-bottom.

    # DATASET_ROOT: folder containing class subfolders (one class per subfolder).
    DATASET_ROOT = "data/class_root_dataset"

    # CROP_NAME: crop key used by taxonomy alignment (e.g., "tomato", "pepper").
    CROP_NAME = "tomato"

    # EPOCHS: total training passes over the train split.
    EPOCHS = 25

    # BATCH_SIZE: samples per optimizer step (increase until near GPU memory limit).
    BATCH_SIZE = 64

    # LEARNING_RATE: optimizer step size for adapter/LoRA parameters.
    LEARNING_RATE = 3e-4

    # LORA_R: LoRA rank (higher = more capacity, more VRAM/compute).
    LORA_R = 16

    # LORA_ALPHA: LoRA scaling factor (commonly 2x to 4x of LORA_R).
    LORA_ALPHA = 32

    # LORA_DROPOUT: dropout applied in LoRA layers (higher = more regularization).
    LORA_DROPOUT = 0.1

    # OOD_FACTOR: multiplier for OOD threshold sensitivity.
    OOD_FACTOR = 2.0

    # NUM_WORKERS: dataloader worker processes (CPU data loading parallelism).
    NUM_WORKERS = 4

    # PREFETCH: dataloader prefetch factor per worker.
    PREFETCH = 2

    # PIN_MEMORY: pin host memory to speed host-to-GPU transfer.
    PIN_MEMORY = True

    # USE_CACHE: keep preprocessed/cached samples in RAM/disk when supported.
    USE_CACHE = False

    # RESUME_MODE: "fresh" starts a new run, "resume" continues latest checkpoint.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # Recommended profile for 100+ GB RAM + A100 GPU (40/80 GB).
    A100_100GB_RECOMMENDED = {
        "EPOCHS": 30,
        "BATCH_SIZE": 128,
        "LEARNING_RATE": 3e-4,
        "LORA_R": 32,
        "LORA_ALPHA": 64,
        "LORA_DROPOUT": 0.05,
        "OOD_FACTOR": 2.0,
        "NUM_WORKERS": 12,
        "PREFETCH": 6,
        "PIN_MEMORY": True,
        "USE_CACHE": True,
    }

    APPLY_A100_100GB_RECOMMENDED = False
    if APPLY_A100_100GB_RECOMMENDED:
        EPOCHS = A100_100GB_RECOMMENDED["EPOCHS"]
        BATCH_SIZE = A100_100GB_RECOMMENDED["BATCH_SIZE"]
        LEARNING_RATE = A100_100GB_RECOMMENDED["LEARNING_RATE"]
        LORA_R = A100_100GB_RECOMMENDED["LORA_R"]
        LORA_ALPHA = A100_100GB_RECOMMENDED["LORA_ALPHA"]
        LORA_DROPOUT = A100_100GB_RECOMMENDED["LORA_DROPOUT"]
        OOD_FACTOR = A100_100GB_RECOMMENDED["OOD_FACTOR"]
        NUM_WORKERS = A100_100GB_RECOMMENDED["NUM_WORKERS"]
        PREFETCH = A100_100GB_RECOMMENDED["PREFETCH"]
        PIN_MEMORY = A100_100GB_RECOMMENDED["PIN_MEMORY"]
        USE_CACHE = A100_100GB_RECOMMENDED["USE_CACHE"]

    BASE_CONFIG = ConfigurationManager(config_dir=str(ROOT / "config"), environment="colab").load_all_configs()
    COLAB_TRAINING_CFG = BASE_CONFIG.get("colab", {}).get("training", {})
    CONTINUAL_DATA_CFG = BASE_CONFIG.get("training", {}).get("continual", {}).get("data", {})
    SEED = int(BASE_CONFIG.get("training", {}).get("continual", {}).get("seed", 42))
    STDOUT_BATCH_INTERVAL = int(COLAB_TRAINING_CFG.get("stdout_progress_batch_interval", 50))
    CHECKPOINT_EVERY_N_STEPS = int(
        COLAB_TRAINING_CFG.get(
            "checkpoint_every_n_steps",
            COLAB_TRAINING_CFG.get("checkpoint_interval", 200),
        )
    )
    CHECKPOINT_ON_EXCEPTION = bool(COLAB_TRAINING_CFG.get("checkpoint_on_exception", True))
    NUM_WORKERS = int(COLAB_TRAINING_CFG.get("num_workers", NUM_WORKERS))
    PIN_MEMORY = bool(COLAB_TRAINING_CFG.get("pin_memory", PIN_MEMORY))
    TARGET_SIZE = int(CONTINUAL_DATA_CFG.get("target_size", 224))
    DATA_SAMPLER = str(CONTINUAL_DATA_CFG.get("sampler", "shuffle"))
    LOADER_ERROR_POLICY = str(CONTINUAL_DATA_CFG.get("loader_error_policy", "tolerant"))
    CACHE_SIZE = int(CONTINUAL_DATA_CFG.get("cache_size", 1000))
    VALIDATE_IMAGES_ON_INIT = bool(CONTINUAL_DATA_CFG.get("validate_images_on_init", True))

    STATE = {
        "validated": False,
        "class_names": [],
        "runtime_dataset_root": None,
        "adapter": None,
        "loaders": None,
        "history": None,
        "calibration": None,
        "checkpoint_manager": CHECKPOINT_MANAGER,
        "resume_manifest": None,
        "best_val_loss": None,
        "best_metric_state": {},
        "telemetry": TELEMETRY,
    }

    if RESUME_MODE == "resume":
        try:
            STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
            if STATE["resume_manifest"]:
                manifest = STATE["resume_manifest"]
                print(
                    f"[RESUME] Found checkpoint: {manifest.get('name', '?')} "
                    f"epoch={manifest.get('epoch', 0)} step={manifest.get('global_step', 0)}"
                )
        except Exception:
            pass

    print(
        f"[CONFIG] crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} "
        f"lr={LEARNING_RATE} resume={RESUME_MODE}"
    )
    TELEMETRY.update_latest(
        {
            "phase": "parameters_ready",
            "crop": CROP_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "resume_mode": RESUME_MODE,
        }
    )

In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Validation"):
    raw_root = Path(DATASET_ROOT).expanduser()
    if not raw_root.is_absolute():
        raw_root = (ROOT / raw_root).resolve()
    if not raw_root.is_dir():
        raise RuntimeError(f"Dataset root not found: {raw_root}")

    class_names = sorted(d.name for d in raw_root.iterdir() if d.is_dir())
    if not class_names:
        raise RuntimeError(f"No class subdirectories in {raw_root}")

    STATE["class_names"] = class_names
    STATE["validated"] = True
    print(f"[DATASET] root={raw_root}  classes={len(class_names)}: {class_names}")
    TELEMETRY.update_latest(
        {
            "phase": "dataset_validated",
            "dataset_root": str(raw_root),
            "class_count": len(class_names),
        }
    )

In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Engine Init"):
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.utils.data_loader import create_training_loaders

    def _normalize(name: str) -> str:
        normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in name.strip())
        while "__" in normalized:
            normalized = normalized.replace("__", "_")
        return normalized.strip("_")

    def _dataset_fingerprint(
        class_root: Path,
        allowed_set: set[str],
        image_exts: set[str],
    ) -> dict[str, int | str]:
        import hashlib

        digest = hashlib.sha256()
        file_count = 0
        total_bytes = 0
        latest_mtime_ns = 0

        class_dirs = sorted(path for path in class_root.iterdir() if path.is_dir())
        for class_dir in class_dirs:
            normalized = _normalize(class_dir.name)
            if not normalized or (allowed_set and normalized not in allowed_set):
                continue

            files = [
                path
                for path in class_dir.rglob("*")
                if path.is_file() and path.suffix.lower() in image_exts
            ]
            for path in sorted(files):
                stats = path.stat()
                relative = path.relative_to(class_root).as_posix()
                digest.update(relative.encode("utf-8"))
                digest.update(b"\0")
                digest.update(str(stats.st_size).encode("ascii"))
                digest.update(b":")
                digest.update(str(stats.st_mtime_ns).encode("ascii"))
                digest.update(b"\n")
                file_count += 1
                total_bytes += int(stats.st_size)
                latest_mtime_ns = max(latest_mtime_ns, int(stats.st_mtime_ns))

        return {
            "file_count": file_count,
            "total_bytes": total_bytes,
            "latest_mtime_ns": latest_mtime_ns,
            "sha256": digest.hexdigest(),
        }

    def prepare_runtime_dataset_layout(
        class_root: Path,
        crop_name: str,
        seed: int = 42,
        allowed: list[str] | None = None,
    ) -> Path:
        """Split class_root into continual/val/test under data/runtime_notebook_datasets/{crop}."""
        runtime_root = ROOT / "data" / "runtime_notebook_datasets"
        crop_root = runtime_root / crop_name
        meta_path = crop_root / "_split_metadata.json"
        image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

        class_dirs = sorted(path for path in class_root.iterdir() if path.is_dir())
        allowed_set = set(allowed or [])
        source_key = {
            "v": 5,
            "root": str(class_root.resolve()),
            "crop": crop_name,
            "seed": seed,
            "dirs": [path.name for path in class_dirs],
            "allowed": sorted(allowed_set),
            "split": "80/10/10",
            "copy_mode": "preserve_relative_path",
            "fingerprint": _dataset_fingerprint(class_root, allowed_set, image_exts),
        }

        if crop_root.exists() and meta_path.exists():
            try:
                if json.loads(meta_path.read_text("utf-8")) == source_key:
                    print("[DATASET] Reusing cached split layout")
                    return runtime_root
            except Exception:
                pass

        if crop_root.exists():
            shutil.rmtree(crop_root)
        crop_root.mkdir(parents=True, exist_ok=True)

        rng = random.Random(seed)
        used = set()
        for class_dir in class_dirs:
            normalized = _normalize(class_dir.name)
            if not normalized or (allowed_set and normalized not in allowed_set) or normalized in used:
                continue
            images = [path for path in class_dir.rglob("*") if path.is_file() and path.suffix.lower() in image_exts]
            rng.shuffle(images)
            image_count = len(images)
            if image_count == 0:
                continue
            train_count = max(1, int(0.8 * image_count))
            val_count = max(1, int(0.1 * image_count)) if image_count >= 3 else 0
            if train_count + val_count >= image_count:
                val_count = max(0, image_count - train_count - 1)
            splits = {
                "continual": images[:train_count],
                "val": images[train_count : train_count + val_count],
                "test": images[train_count + val_count :],
            }
            for split_name, files in splits.items():
                destination_dir = crop_root / split_name / normalized
                destination_dir.mkdir(parents=True, exist_ok=True)
                for source_path in files:
                    relative_path = source_path.relative_to(class_dir)
                    destination_path = destination_dir / relative_path
                    destination_path.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source_path, destination_path)
            used.add(normalized)

        meta_path.write_text(json.dumps(source_key, indent=2), encoding="utf-8")
        return runtime_root

    if not STATE.get("validated"):
        raise RuntimeError("Run dataset validation cell first.")

    crop_name = CROP_NAME.strip().lower()
    class_root = Path(DATASET_ROOT).expanduser()
    if not class_root.is_absolute():
        class_root = (ROOT / class_root).resolve()

    available = sorted({_normalize(path.name) for path in class_root.iterdir() if path.is_dir() and _normalize(path.name)})
    try:
        taxonomy = json.loads((ROOT / "config" / "plant_taxonomy.json").read_text("utf-8"))
        expected = taxonomy.get("crop_specific_diseases", {}).get(crop_name, [])
        if expected:
            aligned = [name for name in sorted({"healthy", *[_normalize(disease) for disease in expected]}) if name in available]
        else:
            aligned = available
    except Exception:
        aligned = available

    if not aligned:
        raise RuntimeError(f"No usable classes for crop '{crop_name}'. Available: {available}")

    STATE["class_names"] = aligned
    print(f"[CLASSES] {aligned}")

    runtime_data_root = prepare_runtime_dataset_layout(
        class_root=class_root,
        crop_name=crop_name,
        allowed=aligned,
    )
    STATE["runtime_dataset_root"] = runtime_data_root

    continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
    continual_cfg["device"] = DEVICE
    continual_cfg["num_epochs"] = EPOCHS
    continual_cfg["batch_size"] = BATCH_SIZE
    continual_cfg["learning_rate"] = LEARNING_RATE
    continual_cfg["adapter"]["lora_r"] = LORA_R
    continual_cfg["adapter"]["lora_alpha"] = LORA_ALPHA
    continual_cfg["adapter"]["lora_dropout"] = LORA_DROPOUT
    continual_cfg["ood"]["threshold_factor"] = OOD_FACTOR

    adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
    print("[ENGINE] Initializing adapter (may download backbone)...")
    adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})

    loader_kwargs = {}
    if NUM_WORKERS > 0:
        loader_kwargs["prefetch_factor"] = PREFETCH

    loaders = create_training_loaders(
        data_dir=str(runtime_data_root),
        crop=crop_name,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        use_cache=USE_CACHE,
        cache_size=CACHE_SIZE,
        target_size=TARGET_SIZE,
        error_policy=LOADER_ERROR_POLICY,
        sampler=DATA_SAMPLER,
        seed=SEED,
        validate_images_on_init=VALIDATE_IMAGES_ON_INIT,
        pin_memory=PIN_MEMORY,
        **loader_kwargs,
    )

    STATE["adapter"] = adapter
    STATE["loaders"] = loaders
    STATE["continual_config"] = continual_cfg

    trainable = sum(parameter.numel() for parameter in adapter.parameters() if parameter.requires_grad)
    print(f"[ENGINE] Ready. trainable_params={trainable:,}  classes={len(aligned)}")
    TELEMETRY.update_latest(
        {
            "phase": "engine_ready",
            "class_count": len(aligned),
            "runtime_dataset_root": str(runtime_data_root),
        }
    )

In [ ]:
with TELEMETRY.capture_cell_output("Cell 6: Training"):
    from scripts.colab_notebook_helpers import (
        build_history_snapshot,
        persist_training_curve_figure,
        persist_training_history_artifacts,
        save_notebook_checkpoint,
    )

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    checkpoint_manager = STATE["checkpoint_manager"]
    val_loader = loaders.get("val")
    run_id = RUN_ID
    telemetry = STATE.get("telemetry") or TELEMETRY

    resume_state = None
    if RESUME_MODE == "resume" and isinstance(STATE.get("resume_manifest"), dict):
        checkpoint_path = str(STATE["resume_manifest"].get("path", "")).strip()
        if checkpoint_path:
            try:
                resume_state = adapter.load_training_checkpoint(checkpoint_path)
                STATE["resume_state"] = resume_state
                progress_state = resume_state.get("progress_state") or {}
                print(
                    f"[RESUME] epoch={progress_state.get('epoch', 0)} "
                    f"step={progress_state.get('global_step', 0)}"
                )
            except Exception as exc:
                print(f"[RESUME] Failed: {exc}")

    existing_history = (resume_state or {}).get("history", (resume_state or {}).get("history_snapshot", {}))
    train_loss_curve = list(existing_history.get("train_loss", []))
    val_loss_curve = list(existing_history.get("val_loss", []))
    val_acc_curve = list(existing_history.get("val_accuracy", []))
    macro_f1_curve = list(existing_history.get("macro_f1", []))
    weighted_f1_curve = list(existing_history.get("weighted_f1", []))
    balanced_acc_curve = list(existing_history.get("balanced_accuracy", []))
    gap_curve = list(existing_history.get("generalization_gap", []))

    start_time = time.time()
    session = None
    last_checkpoint_step = -1
    best_val_loss = float(STATE["best_val_loss"]) if STATE.get("best_val_loss") is not None else None

    print(f"[TRAIN] epochs={EPOCHS} device={DEVICE} batch_interval={STDOUT_BATCH_INTERVAL}")
    telemetry.update_latest(
        {
            "phase": "training_started",
            "epochs": EPOCHS,
            "batch_interval": STDOUT_BATCH_INTERVAL,
        }
    )

    def _history_snapshot():
        return build_history_snapshot(
            state_history=STATE.get("history"),
            train_loss_curve=train_loss_curve,
            val_loss_curve=val_loss_curve,
            val_acc_curve=val_acc_curve,
            macro_f1_curve=macro_f1_curve,
            weighted_f1_curve=weighted_f1_curve,
            balanced_acc_curve=balanced_acc_curve,
            gap_curve=gap_curve,
        )

    def _persist_history():
        snapshot = _history_snapshot()
        STATE["history"] = dict((STATE.get("history") or {}), **snapshot)
        persist_training_history_artifacts(
            root=ROOT,
            history_snapshot=STATE["history"],
            telemetry=telemetry,
        )
        return snapshot

    def _checkpoint(reason, event, mark_best=False, val_loss=None):
        if session is None:
            return None
        record = save_notebook_checkpoint(
            checkpoint_manager=checkpoint_manager,
            adapter=adapter,
            session=session,
            reason=reason,
            run_id=run_id,
            telemetry=telemetry,
            mark_best=bool(mark_best),
            val_loss=(float(val_loss) if val_loss is not None else None),
        )
        if record is not None:
            STATE["resume_manifest"] = record
            print(f"[CKPT] {reason} epoch={record.get('epoch', '?')} step={record.get('global_step', '?')}")
        return record

    def session_observer(record):
        global last_checkpoint_step, best_val_loss
        event_type = record.get("event_type", "")
        event = record.get("payload", {})

        if event_type == "stop_requested":
            return

        if event_type == "batch_end":
            batch_num = int(event.get("batch", 0))
            if batch_num > 0 and (batch_num % STDOUT_BATCH_INTERVAL == 0):
                print(
                    f"[TRAIN] epoch={event.get('epoch', 0)} "
                    f"batch={batch_num}/{event.get('total_batches', 0)} "
                    f"loss={event.get('batch_loss', 0.0):.4f} "
                    f"lr={event.get('lr', 0.0):.6f} "
                    f"speed={event.get('samples_per_sec', 0.0):.1f}s/s "
                    f"elapsed={event.get('elapsed_sec', 0.0):.0f}s eta={event.get('eta_sec', 0.0):.0f}s"
                )
            step = int(event.get("global_step", 0))
            if (
                CHECKPOINT_EVERY_N_STEPS > 0
                and step > 0
                and (step % CHECKPOINT_EVERY_N_STEPS == 0)
                and step != last_checkpoint_step
            ):
                _checkpoint(f"batch_{CHECKPOINT_EVERY_N_STEPS}", event)
                last_checkpoint_step = step

        if event_type == "epoch_end":
            train_loss_curve.append(float(event.get("epoch_loss", 0.0)))
            for key, curve in [
                ("val_loss", val_loss_curve),
                ("val_accuracy", val_acc_curve),
                ("macro_f1", macro_f1_curve),
                ("weighted_f1", weighted_f1_curve),
                ("balanced_accuracy", balanced_acc_curve),
                ("generalization_gap", gap_curve),
            ]:
                if key in event:
                    curve.append(float(event[key]))

            plt.figure(figsize=(13, 3))
            plt.subplot(1, 3, 1)
            plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o", label="Train")
            if val_loss_curve:
                plt.plot(range(1, len(val_loss_curve) + 1), val_loss_curve, marker="s", label="Val")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Loss")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.subplot(1, 3, 2)
            for values, label, marker in [
                (val_acc_curve, "Acc", "^"),
                (macro_f1_curve, "MacroF1", "d"),
                (weighted_f1_curve, "WtdF1", "x"),
                (balanced_acc_curve, "BalAcc", "*"),
            ]:
                if values:
                    plt.plot(range(1, len(values) + 1), values, marker=marker, label=label)
            plt.ylim(0, 1)
            plt.xlabel("Epoch")
            plt.ylabel("Score")
            plt.title("Metrics")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.subplot(1, 3, 3)
            if gap_curve:
                plt.plot(range(1, len(gap_curve) + 1), gap_curve, marker="o", label="Gap")
            plt.axhline(0, color="black", lw=1, alpha=0.5)
            plt.xlabel("Epoch")
            plt.ylabel("Gap")
            plt.title("Gen. Gap")
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.tight_layout()
            persist_training_curve_figure(
                root=ROOT,
                epoch_done=int(event["epoch_done"]),
                telemetry=telemetry,
            )
            plt.close("all")

            _persist_history()

            val_loss = float(event["val_loss"]) if "val_loss" in event else None
            mark_best = False
            if val_loss is not None and (best_val_loss is None or val_loss < best_val_loss):
                best_val_loss = val_loss
                STATE["best_val_loss"] = best_val_loss
                mark_best = True
            _checkpoint("epoch_end", event, mark_best=mark_best, val_loss=val_loss)

            parts = [f"[EPOCH] {event['epoch_done']}/{EPOCHS}: train_loss={event.get('epoch_loss', 0.0):.4f}"]
            if "val_loss" in event:
                parts.append(f"val_loss={event['val_loss']:.4f}")
            if "val_accuracy" in event:
                parts.append(f"val_acc={event['val_accuracy']:.4f}")
            if "macro_f1" in event:
                parts.append(f"macro_f1={event['macro_f1']:.4f}")
            if mark_best:
                parts.append("* BEST")
            print(" ".join(parts))
            telemetry.update_latest(
                {
                    "phase": "training",
                    "epoch_done": int(event["epoch_done"]),
                    "global_step": int(event.get("global_step", 0)),
                    "best_val_loss": best_val_loss,
                }
            )

    session = adapter.build_training_session(
        loaders["train"],
        num_epochs=EPOCHS,
        val_loader=val_loader,
        observers=[session_observer],
        resume_state=resume_state,
        run_id=run_id,
    )

    try:
        history = session.run()
        adapter.is_trained = True
    except Exception as exc:
        print(f"[TRAIN] Exception: {exc}")
        telemetry.emit_log(f"Training exception: {exc}", phase="train", level="error")
        if CHECKPOINT_ON_EXCEPTION:
            try:
                _checkpoint(
                    "exception",
                    {
                        "epoch": 0,
                        "batch": 0,
                        "global_step": int((STATE.get("history") or {}).get("global_step", 0)),
                        "elapsed_sec": time.time() - start_time,
                    },
                )
            except Exception:
                pass
        raise

    elapsed_total = time.time() - start_time
    STATE["history"] = history.to_dict()
    STATE["resume_state"] = session.snapshot_state()
    _persist_history()
    telemetry.update_latest(
        {
            "phase": "training_complete",
            "elapsed_sec": round(elapsed_total, 3),
            "stopped_early": bool(STATE["history"].get("stopped_early", False)),
        }
    )
    print(f"[TRAIN] Complete. elapsed={elapsed_total:.1f}s stopped_early={STATE['history'].get('stopped_early', False)}")

In [ ]:
with TELEMETRY.capture_cell_output("Cell 7: OOD Calibration"):
    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    val_loader = STATE["loaders"].get("val")
    if val_loader is None or len(val_loader.dataset) == 0:
        raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

    calibration = adapter.calibrate_ood(val_loader)
    STATE["calibration"] = calibration

    num_classes = calibration.get("ood_calibration", {}).get("num_classes", 0)
    version = calibration.get("ood_calibration", {}).get("version", 0)
    print(f"[OOD] Calibration done. classes={num_classes} version={version}")
    TELEMETRY.update_latest(
        {
            "phase": "ood_calibrated",
            "ood_num_classes": num_classes,
            "ood_version": version,
        }
    )

In [ ]:
with TELEMETRY.capture_cell_output("Cell 8: Adapter Save"):
    rt("Cell 8: adapter save started", phase="export")

    if STATE.get("adapter") is None:
        raise RuntimeError("Run Cell 5 first.")

    checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    STATE["adapter"].save_adapter(str(checkpoint_dir))
    asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

    print("Saved adapter directory:", asset_dir)
    if not asset_dir.exists():
        raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

    drive_adapter_root = TELEMETRY.artifacts_dir / "adapter_export" / "continual_sd_lora_adapter"
    for path in sorted(asset_dir.rglob("*")):
        if path.is_file():
            relative_path = path.relative_to(checkpoint_dir).as_posix()
            TELEMETRY.copy_artifact_file(path, f"adapter_export/{relative_path}")
            print(" -", path.relative_to(ROOT))

    print("Drive adapter directory:", drive_adapter_root)
    TELEMETRY.update_latest(
        {
            "phase": "adapter_saved",
            "adapter_dir": str(drive_adapter_root),
        }
    )
    rt("Cell 8: adapter save completed", phase="export")

In [ ]:
with TELEMETRY.capture_cell_output("Cell 9: Final Evaluation"):
    from scripts.colab_notebook_helpers import persist_validation_artifacts

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    test_loader = loaders.get("test")
    if test_loader is None or len(test_loader.dataset) == 0:
        raise RuntimeError("Test loader is empty. Final evaluation must use the held-out test split.")

    trainer = adapter._trainer
    if trainer is None:
        raise RuntimeError("Trainer not initialized.")

    trainer.adapter_model.eval()
    trainer.classifier.eval()
    trainer.fusion.eval()

    classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda item: item[1])]

    def _collect_predictions(loader):
        y_true, y_pred = [], []
        with torch.no_grad():
            for batch in loader:
                images = batch["images"].to(trainer.device)
                labels = batch["labels"].to(trainer.device)
                predictions = torch.argmax(trainer.forward_logits(images), dim=1)
                y_true.extend(labels.cpu().tolist())
                y_pred.extend(predictions.cpu().tolist())
        if not y_true:
            raise RuntimeError("No evaluation samples.")
        return y_true, y_pred

    def _evaluate_split(split_name: str, loader, *, artifact_subdir: str, label: str):
        y_true, y_pred = _collect_predictions(loader)
        artifacts = persist_validation_artifacts(
            root=ROOT,
            y_true=y_true,
            y_pred=y_pred,
            classes=classes,
            telemetry=TELEMETRY,
            artifact_subdir=artifact_subdir,
            context={"crop": CROP_NAME, "run_id": RUN_ID, "split_name": split_name},
        )
        accuracy = float(artifacts["report_dict"].get("accuracy", 0.0))
        print(f"[{label}] samples={len(y_true)} classes={len(classes)} accuracy={accuracy:.4f}")
        return artifacts

    results = {}
    val_loader = loaders.get("val")
    if val_loader is not None and len(val_loader.dataset) > 0:
        results["validation"] = _evaluate_split(
            "val",
            val_loader,
            artifact_subdir="validation",
            label="VALIDATION (reference)",
        )

    results["test"] = _evaluate_split(
        "test",
        test_loader,
        artifact_subdir="test",
        label="TEST (held-out)",
    )
    STATE["evaluation_artifacts"] = results
    plt.close("all")

    TELEMETRY.update_latest(
        {
            "phase": "evaluation_complete",
            "evaluation_splits": sorted(results.keys()),
        }
    )
    print("[DONE] Validation reference and held-out test artifacts saved to Drive.")

TELEMETRY.close(
    {
        "status": "ok",
        "evaluation_splits": sorted((STATE.get("evaluation_artifacts") or {}).keys()),
        "cell_outputs_dir": str(TELEMETRY.artifacts_dir / "cell_outputs"),
    }
)